# Entrainer le modele hybride sur Vertex AI Workbench (GCP)

Piste abandonnee : tout a fini par tourner sur Kaggle. Garde pour memoire.

Contrairement a Kaggle et Colab, il n'y a pas de limite de session : l'instance tourne tant
qu'on ne l'arrete pas. C'est aussi le piege, puisqu'elle est facturee a l'heure tant qu'elle
tourne. **Il faut donc penser a l'arreter quand l'entrainement est fini.**

Le code et les donnees viennent d'un bucket GCS, et les checkpoints y sont resynchronises
apres chaque phase.

A faire une fois : creer une instance Workbench avec un GPU, construire le bundle avec
`scripts/zip_selfcontained_colab.py`, et le deposer dans le bucket.

## Config

In [ ]:
# --- GCS (no gs:// prefix on the bucket name) ---
GCS_BUCKET = "YOUR_BUCKET"          # bucket holding the bundle + checkpoints
GCS_PREFIX = "receipt_vlm"          # folder within the bucket
BUNDLE_NAME = "receipt_vlm_colab_bundle.zip"

# --- which phases to run (set False to skip; skipped phases need their checkpoint) ---
RUN_PHASE_1 = True
RUN_PHASE_2 = True
RUN_PHASE_3 = True
RUN_EXPORT  = True

FORCE_SMALL_BATCH = False  # set True on a single small GPU if you hit CUDA OOM (batch 8 -> 4)

GCS_BASE = f"gs://{GCS_BUCKET}/{GCS_PREFIX}"
GCS_CKPT = f"{GCS_BASE}/checkpoints"
assert GCS_BUCKET != "YOUR_BUCKET", "Set GCS_BUCKET to your real bucket name first."
print("Bundle :", f"{GCS_BASE}/{BUNDLE_NAME}")
print("Ckpts  :", GCS_CKPT)

## Le GPU est bien la ?

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU on this Workbench instance — recreate it with a GPU attached"
print(torch.cuda.get_device_name(0))

## Recuperer le code depuis le bucket

Le code est jetable : sur une instance neuve, on le retelecharge. Ce sont les checkpoints qui
comptent, et ils sont dans le bucket.

In [ ]:
import glob, os, subprocess, zipfile
from pathlib import Path

ROOT = Path("/home/jupyter/receipt_vlm")
WORK = ROOT / "repo"
LOCAL_ZIP = ROOT / BUNDLE_NAME

def materialize():
    WORK.mkdir(parents=True, exist_ok=True)
    print("Downloading", f"{GCS_BASE}/{BUNDLE_NAME}", "...")
    subprocess.check_call(["gsutil", "cp", f"{GCS_BASE}/{BUNDLE_NAME}", str(LOCAL_ZIP)])
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(WORK)

if not list(WORK.glob("**/vlm_training/scripts/train.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train.py"), recursive=True)
assert hits, "train.py not found after materialize — wrong bucket/path?"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]   # .../dev_ocr/vlm_training
DEV_OCR = TRAIN_PKG.parent                        # .../dev_ocr
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## Installer les dependances

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## Rediriger les configs vers le disque local
Checkpoints are written to `/home/jupyter/receipt_vlm/checkpoints` and synced to GCS
after each phase (cell 6).

In [ ]:
import yaml
from pathlib import Path

CKPT_DIR = ROOT / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
cfg_path = TRAIN_PKG / "configs" / "colab_paths.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
cfg["checkpoint_dir"] = str(CKPT_DIR)
cfg["log_every"] = 25
if FORCE_SMALL_BATCH:
    cfg["batch_size"] = 4
cfg.setdefault("data", {})
cfg["data"]["real_images_dir"] = str(DEV_OCR / "data" / "raw" / "images_tickets_caisse")
cfg["data"]["real_labels_dir"] = str(TRAIN_PKG / "data" / "real_labels")
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print(cfg_path.read_text())

## Reprendre apres une coupure de session
Pulls any `phase*_best.pt` already in the bucket back to the local disk so you can skip
finished phases (set `RUN_PHASE_1/2 = False` in cell 0). Safe to run on a fresh instance.

In [ ]:
import subprocess
found = []
for name in ("phase1_best.pt", "phase2_best.pt", "phase3_best.pt"):
    dest = CKPT_DIR / name
    rc = subprocess.call(["gsutil", "-q", "cp", f"{GCS_CKPT}/{name}", str(dest)])
    if rc == 0 and dest.is_file():
        found.append(f"{name} ({dest.stat().st_size/1e6:.0f} MB)")
print("Restored:", found if found else f"nothing found in {GCS_CKPT}")

## Entrainer les trois phases

Les checkpoints sont resynchronises vers le bucket apres chaque phase. Compter plusieurs
heures : le demarrage telecharge silencieusement CLIP, SmolLM2, puis CORD.

In [ ]:
import subprocess, sys, os, time, datetime

p1 = f"{CKPT_DIR}/phase1_best.pt"
p2 = f"{CKPT_DIR}/phase2_best.pt"
p3 = f"{CKPT_DIR}/phase3_best.pt"

def gcs_push(local_path):
    """Copy a finished checkpoint up to the bucket so progress survives a stop."""
    if os.path.exists(local_path):
        subprocess.check_call(["gsutil", "-q", "cp", local_path, f"{GCS_CKPT}/"])
        print(f"   synced -> {GCS_CKPT}/{os.path.basename(local_path)}", flush=True)

def run_train(config, out_ckpt, resume=None):
    cmd = [sys.executable, "-u", "scripts/train.py", "--config", config]
    if resume:
        cmd += ["--resume", resume]
    print("\n>>", " ".join(cmd), flush=True)
    print("   (startup is silent for a few min: downloading CLIP+SmolLM2, then CORD)", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    start = last = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        now = time.time()
        gap = now - last; last = now
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{config} failed (exit {proc.returncode})")
    print(f"-- {config} done in {int(time.time()-start)}s", flush=True)
    gcs_push(out_ckpt)

# Skipping a phase requires its checkpoint already on disk (use Restore cell 4b).
if not RUN_PHASE_1 and (RUN_PHASE_2 or RUN_PHASE_3):
    assert os.path.exists(p1), f"{p1} missing -- run cell 4b or set RUN_PHASE_1=True"
if not RUN_PHASE_2 and RUN_PHASE_3:
    assert os.path.exists(p2), f"{p2} missing -- run cell 4b or set RUN_PHASE_2=True"

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml", p1)
if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml", p2, resume=p1)
if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml", p3, resume=p2)
print("Training done")

## Exporter le checkpoint fusionne

In [ ]:
MERGED = f"{CKPT_DIR}/receipt_vlm_500m_merged.pt"
if RUN_EXPORT:
    subprocess.check_call([sys.executable, "scripts/export_checkpoint.py",
                           "--checkpoint", p3, "--output", MERGED])
    print("Merged ->", MERGED)
else:
    print("Export skipped")

## Renvoyer les resultats sur GCS
Pushes the merged model (and any checkpoints) to the bucket. Download them with
`gsutil cp gs://YOUR_BUCKET/receipt_vlm/receipt_vlm_500m_merged.pt .` on your PC.

In [ ]:
from pathlib import Path
import subprocess

if Path(MERGED).is_file():
    subprocess.check_call(["gsutil", "cp", MERGED, f"{GCS_BASE}/"])
    print("Uploaded ->", f"{GCS_BASE}/{Path(MERGED).name}")

print("\nFiles in local checkpoints dir:")
for p in sorted(CKPT_DIR.glob("*")):
    print(f"  {p.name:34} {p.stat().st_size/1e6:8.1f} MB")
print("\nRemember to STOP this Workbench instance to stop billing.")

## Verification rapide sur une photo

In [ ]:
import json, os
from pathlib import Path

photos = sorted((DEV_OCR / "data" / "raw" / "images_tickets_caisse").glob("*.jpg"))
if photos and Path(MERGED).is_file():
    os.environ.update({
        "RECEIPT_OCR_BACKEND": "vlm",
        "RECEIPT_VLM_MODEL": "receipt-vlm-500m",
        "RECEIPT_VLM_MODE": "json",
        "RECEIPT_VLM_MODEL_PATH": MERGED,
    })
    from receipt_ocr import extract_receipt
    print(json.dumps(extract_receipt(str(photos[0])), indent=2, ensure_ascii=False)[:1200])
else:
    print("Need merged checkpoint + at least one photo")